# Ngôn Sắc Việt — Train NLLB Việt ↔ Dân tộc (Google Colab T4 1-Click)

1. Chọn **Runtime → Change runtime type → T4 GPU**.
2. Bấm **Runtime → Run all** (`Ctrl+F9`).
3. Notebook tự kéo code từ GitHub, tải dữ liệu đã rà soát, train câu dài (`epochs=5`, `max_length=256`, `LoRA r=64`), chấm điểm `chrF++ (16+w)`, lưu backup vào Google Drive và tải về `/content/nllb-ethnic-ready.zip`.

In [ ]:
# === CẤU HÌNH CHẠY TỰ ĐỘNG ===
REPO_URL = "https://github.com/minhduy6868/vietlangai-model.git"
BRANCH = "main"
BASE_MODEL = "facebook/nllb-200-distilled-600M"
TRAIN_MODE = "lora"  # "lora" (~180MB, giữ vi<->en gốc) hoặc "full"
RANK = 64
EPOCHS = 5
LR = 3e-4
BATCH = 32
GRAD_ACCUM = 4
MAX_LENGTH = 256
LABEL_SMOOTHING = 0.1
CONCAT_RATIO = 0.35
LONG_UPSAMPLE = 2
TEMP = 2.0
INCLUDE_JARAI = False  # Đổi True nếu đã xác nhận quyền dùng jarai-vie-bible
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/NgonSacViet"


In [ ]:
import shutil
import subprocess
import sys

import torch

smi = shutil.which("nvidia-smi")
if smi:
    print(subprocess.check_output([smi], text=True))
if not torch.cuda.is_available():
    raise RuntimeError("Chưa bật GPU! Chọn Runtime -> Change runtime type -> T4 GPU rồi chạy lại.")
print("CUDA GPU:", torch.cuda.get_device_name(0))

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.56,<5", "peft>=0.17", "accelerate>=1.0",
    "sacrebleu>=2.4", "sentencepiece>=0.2", "pyarrow>=15",
])

DRIVE_READY = False
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_READY = True
        print("Đã kết nối Google Drive:", DRIVE_DIR)
    except Exception as exc:
        print("Không mount được Google Drive (vẫn lưu ở /content):", exc)


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK = Path("/content")
REPO_DIR = WORK / "vietlangai-model"

if (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)])

if DRIVE_READY:
    drive_reviewed = Path(DRIVE_DIR) / "reviewed_pairs.csv"
    if drive_reviewed.is_file():
        dest = REPO_DIR / "data" / "reviewed_pairs.csv"
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_reviewed, dest)
        print("Đã nạp reviewed_pairs.csv từ Google Drive")

os.chdir(REPO_DIR)
print("Working dir:", Path.cwd())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import subprocess
import sys

cmd = [sys.executable, "training/import_public.py"]
if INCLUDE_JARAI:
    cmd += ["--include", "tay-vietnamese-nmt", "rhade-vietnamese-mt", "jarai-vie-bible", "--allow-unknown-license"]
subprocess.check_call(cmd)


In [ ]:
import subprocess
import sys

OUT_DIR = "/content/nllb-ethnic"
train_cmd = [
    sys.executable, "training/finetune_nllb.py",
    "--base", BASE_MODEL,
    "--output", OUT_DIR,
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--batch", str(BATCH),
    "--grad-accum", str(GRAD_ACCUM),
    "--max-length", str(MAX_LENGTH),
    "--label-smoothing", str(LABEL_SMOOTHING),
    "--rank", str(RANK),
    "--concat-ratio", str(CONCAT_RATIO),
    "--long-upsample", str(LONG_UPSAMPLE),
    "--temp", str(TEMP),
]
if INCLUDE_JARAI:
    train_cmd.append("--allow-unknown-license")
if TRAIN_MODE == "full":
    train_cmd.append("--full")

print("Chạy:", " ".join(train_cmd))
subprocess.check_call(train_cmd)


In [ ]:
import subprocess
import sys

eval_cmd = [
    sys.executable, "training/evaluate.py",
    "--base", BASE_MODEL,
    "--adapter", "/content/nllb-ethnic",
    "--split", "test",
    "--batch", "32",
    "--beams", "4",
    "--output", "/content/eval_report.json",
]
if INCLUDE_JARAI:
    eval_cmd.append("--allow-unknown-license")
subprocess.check_call(eval_cmd)


In [ ]:
import sys
sys.path.insert(0, "/content/vietlangai-model")
from app.nllb import NllbEngine

engine = NllbEngine(BASE_MODEL, "/content/nllb-ethnic", beams=4, gated=False)
samples = [
    ("vi", "tyz", "Sáng nay trời mát mẻ, gia đình tôi cùng nhau lên rẫy hái măng. Buổi chiều chúng tôi quay về bản nấu cơm và trò chuyện bên bếp lửa."),
    ("vi", "rad", "Hôm nay buôn làng tổ chức lễ hội đầu mùa rất vui vẻ. Mọi người mặc trang phục truyền thống và cùng nhau đánh chiêng đón khách quý."),
]
for src, tgt, text in samples:
    if (src, tgt) in engine.pairs():
        out = engine.translate(text, src, tgt)
        back = engine.translate(out, tgt, src)
        print(f"[{src}->{tgt}] {out}")
        print(f"[{tgt}->{src}] {back}\n")


In [ ]:
import shutil
from pathlib import Path

bundle_dir = Path("/content/nllb-ethnic")
ckpt_dir = bundle_dir / "checkpoints"
if ckpt_dir.exists():
    shutil.rmtree(ckpt_dir)

zip_file = shutil.make_archive("/content/nllb-ethnic-ready", "zip", root_dir="/content", base_dir="nllb-ethnic")
print("Đã đóng gói:", zip_file)

if DRIVE_READY:
    drive_out = Path(DRIVE_DIR)
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_file, drive_out / "nllb-ethnic-ready.zip")
    eval_json = Path("/content/eval_report.json")
    if eval_json.is_file():
        shutil.copy2(eval_json, drive_out / "eval_report.json")
    print("Đã sao lưu lên Google Drive:", drive_out)

try:
    from google.colab import files
    files.download(zip_file)
except Exception:
    pass
